# Local LLM agents: tool-calling reliability benchmark

Runs a held-out suite of **72 test tasks** (plus 12 dev tasks) with known-correct tool calls against
local open-weight models served by **Ollama**, in three modes:

* `single_shot`: one model reply; every needed tool call must be in it
* `react`: LangGraph ReAct loop; schema errors are fed back to the model
* `react_no_error_feedback`: same loop, but the first bad call ends the run (ablation)

Models: `llama3.2:3b`, `qwen2.5:7b`, `qwen2.5:3b`, `llama3.2:1b`. Temperature 0, seed 42, context 4096, recursion limit 25.

**How to run on Kaggle**
1. *Settings → Accelerator → GPU T4 x2* (or P100). *Settings → Internet → On* (needs a phone-verified account).
2. Click **Save Version → Save & Run All (Commit)**. It runs in the background; you can close the tab.
3. When it finishes, open the version → **Output** → download `results.zip`
   (`summary.md`, `scores.csv`, `traces.jsonl`, `examples.md`, `env_*.json`).

The run is resumable: if it stops, re-running skips tasks already in `traces.jsonl`.

In [ ]:
import os, sys, subprocess, time, json, urllib.request
from pathlib import Path
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
os.chdir(WORK)
MODELS = ["llama3.2:3b", "qwen2.5:7b", "qwen2.5:3b", "llama3.2:1b"]
MODELS_STR = " ".join(MODELS)
OUT = WORK / "results"
print("work dir:", WORK)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install and start Ollama

In [ ]:
!apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y zstd pciutils lshw >/dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3
!ollama --version

In [ ]:
log = open(WORK / "ollama_server.log", "w")
server = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT,
                          env={**os.environ, "OLLAMA_KEEP_ALIVE": "30m", "OLLAMA_HOST": "127.0.0.1:11434"})
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode()); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start; see ollama_server.log")

In [ ]:
for m in MODELS:
    t0 = time.time()
    subprocess.run(["ollama", "pull", m], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"pulled {m} in {time.time()-t0:.0f}s")
!ollama list

## 2. Python dependencies (pinned to the versions the harness was tested with)

In [ ]:
!pip install -q "langgraph==1.2.23" "langchain-core==1.8.3" "langchain-ollama==1.1.0" "ollama==0.6.3" pytest

## 3. Write the harness source

In [ ]:
!mkdir -p reliability_eval

In [ ]:
%%writefile reliability_eval/__init__.py
"""Tool-calling reliability benchmark for local LLM agents (LangGraph + Ollama)."""

In [ ]:
%%writefile reliability_eval/__main__.py
from .run_eval import main

main()

In [ ]:
%%writefile reliability_eval/tools.py
"""Tool registry for the reliability benchmark.

Every tool is deterministic and offline (mock data), so a run is reproducible:
the same call always returns the same result, and the only source of
variation is the model.

Nine tools in four groups:
  arithmetic : add, subtract, multiply, divide
  weather    : get_weather (enum argument `unit`)
  currency   : convert_currency (fixed mock rates)
  employees  : find_employee (name -> id), get_employee (id -> record)
  reminders  : create_reminder (free-text + integer minutes)
"""

from __future__ import annotations

from typing import Literal

from langchain_core.tools import tool

# --------------------------------------------------------------------------
# Mock data
# --------------------------------------------------------------------------

WEATHER_C = {  # temperature in Celsius + condition
    "mumbai": (31, "humid and partly cloudy"),
    "pune": (27, "clear"),
    "delhi": (22, "hazy"),
    "bengaluru": (24, "light drizzle"),
    "london": (9, "rainy"),
    "new york": (12, "windy"),
    "tokyo": (15, "sunny"),
    "paris": (11, "overcast"),
}
CITY_ALIASES = {"bangalore": "bengaluru", "bombay": "mumbai", "nyc": "new york",
                "new york city": "new york", "new delhi": "delhi"}

# Units of currency per 1 USD (fixed, fictional snapshot)
RATES_PER_USD = {"USD": 1.0, "EUR": 0.92, "GBP": 0.79, "INR": 83.0, "JPY": 150.0}

EMPLOYEES = {
    1000: {"name": "Vikram Mehta", "department": "Engineering", "office_city": "Bengaluru", "manager_id": None},
    1001: {"name": "Ananya Rao", "department": "Engineering", "office_city": "Bengaluru", "manager_id": 1000},
    1007: {"name": "Rahul Verma", "department": "Finance", "office_city": "Mumbai", "manager_id": None},
    1013: {"name": "Arjun Nair", "department": "Sales", "office_city": "Delhi", "manager_id": 1021},
    1021: {"name": "Meera Iyer", "department": "Sales", "office_city": "London", "manager_id": None},
    1042: {"name": "Priya Shah", "department": "Finance", "office_city": "Pune", "manager_id": 1007},
    1050: {"name": "Kabir Singh", "department": "Research", "office_city": "Tokyo", "manager_id": 1000},
    1066: {"name": "Sara Thomas", "department": "Legal", "office_city": "New York", "manager_id": 1007},
}


def _norm_city(city: str) -> str:
    c = " ".join(city.lower().replace(",", " ").split())
    c = CITY_ALIASES.get(c, c)
    for known in WEATHER_C:
        if known in c:
            return known
    return c


# --------------------------------------------------------------------------
# Tools (docstrings are what the model reads)
# --------------------------------------------------------------------------

@tool
def add(a: float, b: float) -> float:
    """Add two numbers and return a + b.

    Args:
        a: First number.
        b: Second number.
    """
    return a + b


@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a and return a - b.

    Args:
        a: The number to subtract from.
        b: The number to subtract.
    """
    return a - b


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return a * b.

    Args:
        a: First number.
        b: Second number.
    """
    return a * b


@tool
def divide(a: float, b: float) -> float:
    """Divide a by b and return a / b.

    Args:
        a: The dividend.
        b: The divisor (must not be zero).
    """
    if b == 0:
        raise ValueError("Division by zero is not allowed.")
    return a / b


@tool
def get_weather(city: str, unit: Literal["celsius", "fahrenheit"] = "celsius") -> dict:
    """Get the current weather for a city.

    Args:
        city: City name, e.g. "Pune" or "London".
        unit: Temperature unit, either "celsius" or "fahrenheit". Defaults to "celsius".
    """
    key = _norm_city(city)
    if key not in WEATHER_C:
        return {"city": city, "available": False, "message": f"No weather data for {city}."}
    t_c, cond = WEATHER_C[key]
    temp = t_c if unit == "celsius" else round(t_c * 9 / 5 + 32, 1)
    return {"city": key.title(), "temperature": temp, "unit": unit, "condition": cond}


@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    """Convert an amount of money from one currency to another.

    Args:
        amount: The amount of money to convert.
        from_currency: Three-letter ISO code of the source currency (USD, EUR, GBP, INR, JPY).
        to_currency: Three-letter ISO code of the target currency (USD, EUR, GBP, INR, JPY).
    """
    f, t = from_currency.strip().upper(), to_currency.strip().upper()
    for code in (f, t):
        if code not in RATES_PER_USD:
            raise ValueError(
                f"Unknown currency code '{code}'. Use one of {sorted(RATES_PER_USD)}."
            )
    converted = round(amount / RATES_PER_USD[f] * RATES_PER_USD[t], 2)
    return {"amount": amount, "from": f, "to": t, "converted": converted}


@tool
def find_employee(name: str) -> list:
    """Search the employee directory by (part of) a name and return matching employee IDs.

    Args:
        name: Full or partial employee name, e.g. "Priya" or "Priya Shah".
    """
    q = name.strip().lower()
    hits = [{"employee_id": eid, "name": rec["name"]}
            for eid, rec in EMPLOYEES.items() if q and q in rec["name"].lower()]
    return hits


@tool
def get_employee(employee_id: int) -> dict:
    """Get an employee's record (name, department, office city, manager ID) by employee ID.

    Args:
        employee_id: Numeric employee ID, e.g. 1042.
    """
    rec = EMPLOYEES.get(int(employee_id))
    if rec is None:
        raise ValueError(f"No employee with ID {employee_id}.")
    return {"employee_id": int(employee_id), **rec}


@tool
def create_reminder(text: str, minutes: int) -> dict:
    """Create a reminder that fires after a number of minutes from now.

    Args:
        text: What to remind the user about.
        minutes: Minutes from now until the reminder fires (whole number).
    """
    if minutes <= 0:
        raise ValueError("minutes must be a positive whole number.")
    return {"status": "scheduled", "text": text, "minutes": minutes}


TOOLS = [add, subtract, multiply, divide, get_weather, convert_currency,
         find_employee, get_employee, create_reminder]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

# Tools whose two numeric arguments may be swapped without changing the result.
COMMUTATIVE = {"add": ("a", "b"), "multiply": ("a", "b")}


def tool_schema(name: str) -> dict:
    """JSON schema of a tool's arguments: {'properties': ..., 'required': [...]}"""
    return TOOLS_BY_NAME[name].args_schema.model_json_schema()


def run_tool(name: str, args: dict):
    """Execute a tool by name (used to build ground truth). Raises on error."""
    return TOOLS_BY_NAME[name].invoke(args)

In [ ]:
%%writefile reliability_eval/tasks.py
"""Ground-truth task suite.

Each task = a user prompt + the known-correct tool calls (name + arguments)
+ the expected final answer.

Splits
  dev  (12 tasks): used only to smoke-test the harness and freeze the system
                   prompt BEFORE the test split is run.
  test (72 tasks): held out. Never used to change the prompt, tools or
                   matching rules. All reported numbers come from this split.

Categories
  single_simple : one call, arguments stated plainly in the prompt
  single_args   : one call, but the argument needs care (enum value, unit
                  conversion like "2 hours" -> 120, number words, order)
  multi_step    : 2-3 calls; most are dependent (call 2 needs call 1's result),
                  a few are independent pairs (order does not matter)
  no_tool       : general knowledge; the correct behaviour is NO tool call

Argument matching spec (see scoring.match_value)
  plain number            -> numeric match, relative tolerance 1e-3
  plain string            -> case-insensitive, whitespace-normalised equality
  {"$contains": [kw,...]} -> string must contain every keyword (case-insensitive)
  {"$any": [v1, v2,...]}  -> any one of the listed values is accepted
"""

from __future__ import annotations

import json
from pathlib import Path

C = lambda *kw: {"$contains": list(kw)}  # noqa: E731
ANY = lambda *v: {"$any": list(v)}  # noqa: E731


def T(tid, split, cat, prompt, calls=(), answer=None, ordered=True):
    return {
        "id": tid, "split": split, "category": cat, "prompt": prompt,
        "expected_calls": [{"name": n, "args": a} for n, a in calls],
        "ordered": ordered,
        # expected_answer: number, string, list of acceptable strings (any),
        # or {"$all": [...]} (every item must appear). None = not checked.
        "expected_answer": answer,
    }


TASKS = [
    # ------------------------------------------------------------- dev (12)
    T("dev_s1", "dev", "single_simple", "What is 56 plus 44?", [("add", {"a": 56, "b": 44})], 100),
    T("dev_s2", "dev", "single_simple", "What's the weather in Mumbai?", [("get_weather", {"city": "Mumbai"})], 31),
    T("dev_s3", "dev", "single_simple", "Convert 10 dollars to euros.",
      [("convert_currency", {"amount": 10, "from_currency": "USD", "to_currency": "EUR"})], 9.2),
    T("dev_a1", "dev", "single_args", "Weather in Tokyo in fahrenheit.",
      [("get_weather", {"city": "Tokyo", "unit": "fahrenheit"})], 59),
    T("dev_a2", "dev", "single_args", "Remind me in 30 minutes to drink water.",
      [("create_reminder", {"text": C("water"), "minutes": 30})], None),
    T("dev_a3", "dev", "single_args", "What's nine times eight?", [("multiply", {"a": 9, "b": 8})], 72),
    T("dev_m1", "dev", "multi_step", "Add 10 and 15, then multiply the result by 4.",
      [("add", {"a": 10, "b": 15}), ("multiply", {"a": 25, "b": 4})], 100),
    T("dev_m2", "dev", "multi_step", "What department is Ananya Rao in?",
      [("find_employee", {"name": C("ananya")}), ("get_employee", {"employee_id": 1001})], "Engineering"),
    T("dev_m3", "dev", "multi_step", "Get the weather in London and in Paris.",
      [("get_weather", {"city": "London"}), ("get_weather", {"city": "Paris"})], {"$all": [9, 11]}, ordered=False),
    T("dev_n1", "dev", "no_tool", "What is the capital of Japan?", [], "Tokyo"),
    T("dev_n2", "dev", "no_tool", "Who painted the Mona Lisa?", [], ["Leonardo", "da Vinci"]),
    T("dev_n3", "dev", "no_tool", "How many continents are there on Earth?", [], ["7", "seven"]),

    # -------------------------------------------------- test: single_simple (22)
    T("s01", "test", "single_simple", "What is 25 times 4?", [("multiply", {"a": 25, "b": 4})], 100),
    T("s02", "test", "single_simple", "Add 348 and 1276.", [("add", {"a": 348, "b": 1276})], 1624),
    T("s03", "test", "single_simple", "What's 7391 multiplied by 842?", [("multiply", {"a": 7391, "b": 842})], 6223222),
    T("s04", "test", "single_simple", "Subtract 1999 from 5000.", [("subtract", {"a": 5000, "b": 1999})], 3001),
    T("s05", "test", "single_simple", "What is 144 divided by 12?", [("divide", {"a": 144, "b": 12})], 12),
    T("s06", "test", "single_simple", "How much is 3.75 plus 8.6?", [("add", {"a": 3.75, "b": 8.6})], 12.35),
    T("s07", "test", "single_simple", "What's the weather in Pune right now?", [("get_weather", {"city": "Pune"})], 27),
    T("s08", "test", "single_simple", "Is it cold in London today?", [("get_weather", {"city": "London"})], 9),
    T("s09", "test", "single_simple", "Tell me the current weather for Tokyo.", [("get_weather", {"city": "Tokyo"})], 15),
    T("s10", "test", "single_simple", "Convert 100 euros to Japanese yen.",
      [("convert_currency", {"amount": 100, "from_currency": "EUR", "to_currency": "JPY"})], 16304.35),
    T("s11", "test", "single_simple", "How many Indian rupees is 250 US dollars?",
      [("convert_currency", {"amount": 250, "from_currency": "USD", "to_currency": "INR"})], 20750),
    T("s12", "test", "single_simple", "Change 5000 Indian rupees into British pounds.",
      [("convert_currency", {"amount": 5000, "from_currency": "INR", "to_currency": "GBP"})], 47.59),
    T("s13", "test", "single_simple", "Look up employee 1042.", [("get_employee", {"employee_id": 1042})], "Priya Shah"),
    T("s14", "test", "single_simple", "What department does employee 1013 work in?",
      [("get_employee", {"employee_id": 1013})], "Sales"),
    T("s15", "test", "single_simple", "Find the employee ID of Kabir Singh.",
      [("find_employee", {"name": C("kabir")})], 1050),
    T("s16", "test", "single_simple", "Remind me to call the dentist in 45 minutes.",
      [("create_reminder", {"text": C("dentist"), "minutes": 45})], None),
    T("s17", "test", "single_simple", "Set a reminder in 10 minutes to check the oven.",
      [("create_reminder", {"text": C("oven"), "minutes": 10})], None),
    T("s18", "test", "single_simple", "What is 1000 minus 1?", [("subtract", {"a": 1000, "b": 1})], 999),
    T("s19", "test", "single_simple", "Divide 7 by 2.", [("divide", {"a": 7, "b": 2})], 3.5),
    T("s20", "test", "single_simple", "What's 0.15 times 640?", [("multiply", {"a": 0.15, "b": 640})], 96),
    T("s21", "test", "single_simple", "What's the weather like in New York?",
      [("get_weather", {"city": ANY("New York", "New York City", "NYC")})], 12),
    T("s22", "test", "single_simple", "Who is employee number 1001?", [("get_employee", {"employee_id": 1001})], "Ananya Rao"),

    # ---------------------------------------------------- test: single_args (14)
    T("a01", "test", "single_args", "What's the temperature in Mumbai in Fahrenheit?",
      [("get_weather", {"city": "Mumbai", "unit": "fahrenheit"})], 87.8),
    T("a02", "test", "single_args", "Give me Delhi's weather in fahrenheit please.",
      [("get_weather", {"city": ANY("Delhi", "New Delhi"), "unit": "fahrenheit"})], 71.6),
    T("a03", "test", "single_args", "Weather in Paris, in Celsius.",
      [("get_weather", {"city": "Paris", "unit": "celsius"})], 11),
    T("a04", "test", "single_args", "Remind me in 2 hours to submit the report.",
      [("create_reminder", {"text": C("report"), "minutes": 120})], None),
    T("a05", "test", "single_args", "Set a reminder for one and a half hours from now: pick up groceries.",
      [("create_reminder", {"text": C("groceries"), "minutes": 90})], None),
    T("a06", "test", "single_args", "In a quarter of an hour, remind me to stretch.",
      [("create_reminder", {"text": C("stretch"), "minutes": 15})], None),
    T("a07", "test", "single_args", "What is twelve times thirteen?", [("multiply", {"a": 12, "b": 13})], 156),
    T("a08", "test", "single_args", "Add negative 45 and 120.", [("add", {"a": -45, "b": 120})], 75),
    T("a09", "test", "single_args", "Convert 1,250.50 GBP to USD.",
      [("convert_currency", {"amount": 1250.5, "from_currency": "GBP", "to_currency": "USD"})], 1582.91),
    T("a10", "test", "single_args", "How many US dollars do I get for 10000 yen?",
      [("convert_currency", {"amount": 10000, "from_currency": "JPY", "to_currency": "USD"})], 66.67),
    T("a11", "test", "single_args", "Take 72 away from 300.", [("subtract", {"a": 300, "b": 72})], 228),
    T("a12", "test", "single_args", "Split a bill of 1350 rupees equally among 6 people. How much does each pay?",
      [("divide", {"a": 1350, "b": 6})], 225),
    T("a13", "test", "single_args", "Which city is employee 1066's office in?",
      [("get_employee", {"employee_id": 1066})], "New York"),
    T("a14", "test", "single_args", "Search the directory for an employee named Meera.",
      [("find_employee", {"name": C("meera")})], 1021),

    # ----------------------------------------------------- test: multi_step (20)
    T("m01", "test", "multi_step", "What is 15 plus 27, then multiply the result by 3?",
      [("add", {"a": 15, "b": 27}), ("multiply", {"a": 42, "b": 3})], 126),
    T("m02", "test", "multi_step", "Add 120 and 80, then divide the total by 4.",
      [("add", {"a": 120, "b": 80}), ("divide", {"a": 200, "b": 4})], 50),
    T("m03", "test", "multi_step", "Multiply 6 by 7 and then subtract 2 from the result.",
      [("multiply", {"a": 6, "b": 7}), ("subtract", {"a": 42, "b": 2})], 40),
    T("m04", "test", "multi_step", "Subtract 250 from 1000 and then multiply what's left by 2.",
      [("subtract", {"a": 1000, "b": 250}), ("multiply", {"a": 750, "b": 2})], 1500),
    T("m05", "test", "multi_step", "First add 18 and 24, then divide that sum by 7.",
      [("add", {"a": 18, "b": 24}), ("divide", {"a": 42, "b": 7})], 6),
    T("m06", "test", "multi_step", "Add 20 and 30 US dollars, then convert the total to Indian rupees.",
      [("add", {"a": 20, "b": 30}),
       ("convert_currency", {"amount": 50, "from_currency": "USD", "to_currency": "INR"})], 4150),
    T("m07", "test", "multi_step", "Convert 200 euros to US dollars, then add 15 to the converted amount.",
      [("convert_currency", {"amount": 200, "from_currency": "EUR", "to_currency": "USD"}),
       ("add", {"a": 217.39, "b": 15})], 232.39),
    T("m08", "test", "multi_step", "What is the weather in the city where employee 1042's office is?",
      [("get_employee", {"employee_id": 1042}), ("get_weather", {"city": "Pune"})], 27),
    T("m09", "test", "multi_step", "Find Priya Shah's employee ID and then tell me her department.",
      [("find_employee", {"name": C("priya")}), ("get_employee", {"employee_id": 1042})], "Finance"),
    T("m10", "test", "multi_step", "Who is the manager of employee 1013? Give me the manager's name.",
      [("get_employee", {"employee_id": 1013}), ("get_employee", {"employee_id": 1021})], "Meera Iyer"),
    T("m11", "test", "multi_step", "What's the weather in the office city of Kabir Singh?",
      [("find_employee", {"name": C("kabir")}), ("get_employee", {"employee_id": 1050}),
       ("get_weather", {"city": "Tokyo"})], 15),
    T("m12", "test", "multi_step", "Look up Arjun Nair's manager and tell me which department the manager is in.",
      [("find_employee", {"name": C("arjun")}), ("get_employee", {"employee_id": 1013}),
       ("get_employee", {"employee_id": 1021})], "Sales"),
    T("m13", "test", "multi_step", "Multiply 12 by 12, then add 56 to the result.",
      [("multiply", {"a": 12, "b": 12}), ("add", {"a": 144, "b": 56})], 200),
    T("m14", "test", "multi_step", "Divide 900 by 3, then subtract 45 from the result.",
      [("divide", {"a": 900, "b": 3}), ("subtract", {"a": 300, "b": 45})], 255),
    T("m15", "test", "multi_step", "Get the current weather in Bengaluru and in Delhi.",
      [("get_weather", {"city": ANY("Bengaluru", "Bangalore")}), ("get_weather", {"city": ANY("Delhi", "New Delhi")})],
      {"$all": [24, 22]}, ordered=False),
    T("m16", "test", "multi_step", "Convert 50 British pounds to euros, and also 50 British pounds to yen.",
      [("convert_currency", {"amount": 50, "from_currency": "GBP", "to_currency": "EUR"}),
       ("convert_currency", {"amount": 50, "from_currency": "GBP", "to_currency": "JPY"})],
      {"$all": [58.23, 9493.67]}, ordered=False),
    T("m17", "test", "multi_step", "What's 45 times 3, and separately, what is 210 divided by 7?",
      [("multiply", {"a": 45, "b": 3}), ("divide", {"a": 210, "b": 7})], {"$all": [135, 30]}, ordered=False),
    T("m18", "test", "multi_step", "Look up employees 1000 and 1007 and tell me their names.",
      [("get_employee", {"employee_id": 1000}), ("get_employee", {"employee_id": 1007})],
      {"$all": ["Vikram Mehta", "Rahul Verma"]}, ordered=False),
    T("m19", "test", "multi_step", "What is the temperature in Sara Thomas's office city, in Fahrenheit?",
      [("find_employee", {"name": C("sara")}), ("get_employee", {"employee_id": 1066}),
       ("get_weather", {"city": ANY("New York", "New York City", "NYC"), "unit": "fahrenheit"})], 53.6),
    T("m20", "test", "multi_step", "Add 5 and 7, multiply the result by 10, then divide that by 4.",
      [("add", {"a": 5, "b": 7}), ("multiply", {"a": 12, "b": 10}), ("divide", {"a": 120, "b": 4})], 30),

    # -------------------------------------------------------- test: no_tool (16)
    T("n01", "test", "no_tool", "What is the capital of France?", [], "Paris"),
    T("n02", "test", "no_tool", "Who wrote Romeo and Juliet?", [], "Shakespeare"),
    T("n03", "test", "no_tool", "How many days are there in a week?", [], ["7", "seven"]),
    T("n04", "test", "no_tool", "What is the chemical symbol for gold?", [], "Au"),
    T("n05", "test", "no_tool", "Translate 'thank you' into Spanish.", [], "gracias"),
    T("n06", "test", "no_tool", "What does CPU stand for?", [], "central processing unit"),
    T("n07", "test", "no_tool", "Which planet is known as the Red Planet?", [], "Mars"),
    T("n08", "test", "no_tool", "Say hello to me in a friendly way.", [], None),
    T("n09", "test", "no_tool", "At sea level, what is the boiling point of water in Celsius?", [], "100"),
    T("n10", "test", "no_tool", "What kinds of things can you help me with?", [], None),
    T("n11", "test", "no_tool", "Which ocean is the largest on Earth?", [], "Pacific"),
    T("n12", "test", "no_tool", "What's the opposite of the word 'hot'?", [], "cold"),
    T("n13", "test", "no_tool", "Give me one synonym for 'happy'.", [], None),
    T("n14", "test", "no_tool", "What language is mainly spoken in Brazil?", [], "Portuguese"),
    T("n15", "test", "no_tool", "Botanically speaking, is a tomato a fruit or a vegetable?", [], "fruit"),
    T("n16", "test", "no_tool", "What is the largest mammal in the world?", [], "blue whale"),
]


def load_tasks(split: str | None = None) -> list[dict]:
    if split in (None, "all"):
        return list(TASKS)
    return [t for t in TASKS if t["split"] == split]


def write_jsonl(path: str | Path) -> None:
    with open(path, "w") as f:
        for t in TASKS:
            f.write(json.dumps(t) + "\n")


if __name__ == "__main__":
    import collections
    out = Path(__file__).with_name("tasks.jsonl")
    write_jsonl(out)
    c = collections.Counter((t["split"], t["category"]) for t in TASKS)
    for k, v in sorted(c.items()):
        print(k, v)
    print("total", len(TASKS), "->", out)

In [ ]:
%%writefile reliability_eval/scoring.py
"""Validation of tool-call attempts, matching against ground truth, and the
rule-based failure taxonomy.

Vocabulary
  attempt   : one tool call the model tried to make (structured, unparseable,
              or written as JSON text in the reply instead of a real call)
  status    : what happened to the attempt
                ok               valid against the schema and executed fine
                tool_error       schema-valid, but the tool rejected the value
                                 (e.g. currency "dollars", employee 9999)
                unknown_tool     tool name not in the registry
                unknown_args     argument key not in the tool's schema
                missing_args     a required argument is absent
                validation_error value cannot be coerced to the schema type /
                                 enum (e.g. unit="F", minutes="soon")
                parse_error      call could not be parsed as a structured call
  MALFORMED = unknown_tool | unknown_args | missing_args | validation_error | parse_error
"""

from __future__ import annotations

import itertools
import json
import math
import re
from collections import Counter

from pydantic import ValidationError

from .tools import COMMUTATIVE, TOOLS_BY_NAME, tool_schema

MALFORMED = {"unknown_tool", "unknown_args", "missing_args", "validation_error", "parse_error"}

NUM_REL_TOL = 1e-3      # argument numbers: 0.1 % relative tolerance
ANSWER_REL_TOL = 5e-3   # numbers in the final answer: 0.5 % (allows rounding to 2 dp)


# --------------------------------------------------------------------------
# Validation of one structured call
# --------------------------------------------------------------------------

def _json_type_ok(prop: dict, value) -> bool:
    """Is the RAW value the JSON type the schema asks for (before coercion)?"""
    t = prop.get("type")
    if t is None and "anyOf" in prop:
        return any(_json_type_ok(p, value) for p in prop["anyOf"])
    if t == "number":
        return isinstance(value, (int, float)) and not isinstance(value, bool)
    if t == "integer":
        return (isinstance(value, int) and not isinstance(value, bool)) or (
            isinstance(value, float) and value.is_integer())
    if t == "string":
        return isinstance(value, str)
    return True


def validate_call(name, raw_args) -> dict:
    """Check a structured call against the registry. Does not execute it.

    Returns {status, error, args (coerced, if valid), type_drift: [keys]}.
    """
    out = {"status": "ok", "error": None, "args": None, "type_drift": []}
    if name not in TOOLS_BY_NAME:
        out.update(status="unknown_tool",
                   error=f"Unknown tool '{name}'. Available tools: {sorted(TOOLS_BY_NAME)}.")
        return out
    if not isinstance(raw_args, dict):
        out.update(status="parse_error", error=f"Arguments must be a JSON object, got {raw_args!r}.")
        return out
    schema = tool_schema(name)
    props = schema.get("properties", {})
    unknown = [k for k in raw_args if k not in props]
    if unknown:
        out.update(status="unknown_args",
                   error=f"Unknown argument(s) {unknown} for '{name}'. Valid arguments: {list(props)}.")
        return out
    out["type_drift"] = [k for k, v in raw_args.items() if not _json_type_ok(props[k], v)]
    try:
        model = TOOLS_BY_NAME[name].args_schema.model_validate(raw_args)
    except ValidationError as e:
        missing = [".".join(map(str, err["loc"])) for err in e.errors() if err["type"] == "missing"]
        if missing:
            out.update(status="missing_args",
                       error=f"Missing required argument(s) {missing} for '{name}'.")
        else:
            msgs = "; ".join(f"{'.'.join(map(str, err['loc']))}: {err['msg']}" for err in e.errors())
            out.update(status="validation_error", error=f"Invalid arguments for '{name}': {msgs}")
        return out
    out["args"] = {k: v for k, v in model.model_dump().items() if k in props}
    return out


def execute_call(name: str, args: dict) -> tuple[str, object, str | None]:
    """Run a validated call. Returns (status, result, error)."""
    try:
        return "ok", TOOLS_BY_NAME[name].invoke(args), None
    except Exception as e:  # tool rejected the value
        return "tool_error", None, f"Tool '{name}' failed: {e}"


# --------------------------------------------------------------------------
# Detecting tool calls written as text (a parsing failure)
# --------------------------------------------------------------------------

_TEXT_CALL_PATTERNS = [
    re.compile(r'"name"\s*:\s*"([A-Za-z_]\w*)"'),
    re.compile(r'"function"\s*:\s*"([A-Za-z_]\w*)"'),
    re.compile(r"<tool_call>\s*\{.*?\"name\"\s*:\s*\"([A-Za-z_]\w*)\"", re.S),
]


def text_json_calls(content: str) -> list[str]:
    """Names of tool calls the model wrote into its reply text instead of
    emitting a structured call. Only names that look like tools count."""
    if not content:
        return []
    names = []
    for pat in _TEXT_CALL_PATTERNS:
        for m in pat.finditer(content):
            n = m.group(1)
            if n in TOOLS_BY_NAME and n not in names:
                names.append(n)
    return names


# --------------------------------------------------------------------------
# Matching against ground truth
# --------------------------------------------------------------------------

def _norm_str(s: str) -> str:
    return " ".join(str(s).lower().strip().strip(".").split())


def _to_num(v):
    if isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        try:
            return float(v.replace(",", "").strip())
        except ValueError:
            return None
    return None


def match_value(spec, value) -> bool:
    if isinstance(spec, dict) and "$contains" in spec:
        return isinstance(value, str) and all(k.lower() in value.lower() for k in spec["$contains"])
    if isinstance(spec, dict) and "$any" in spec:
        return any(match_value(s, value) for s in spec["$any"])
    if spec is None:
        return value is None
    if isinstance(spec, (int, float)) and not isinstance(spec, bool):
        v = _to_num(value)
        return v is not None and math.isclose(v, float(spec), rel_tol=NUM_REL_TOL, abs_tol=1e-6)
    return isinstance(value, str) and _norm_str(value) == _norm_str(spec)


def match_args(tool: str, expected: dict, got: dict) -> bool:
    def _plain(exp, g):
        for k, spec in exp.items():
            if not match_value(spec, g.get(k)):
                return False
        props = tool_schema(tool).get("properties", {})
        for k, v in g.items():  # extra args must equal their schema default
            if k not in exp and v != props.get(k, {}).get("default", object()):
                return False
        return True

    if _plain(expected, got):
        return True
    if tool in COMMUTATIVE:
        x, y = COMMUTATIVE[tool]
        swapped = dict(got)
        swapped[x], swapped[y] = got.get(y), got.get(x)
        return _plain(expected, swapped)
    return False


def _dedupe(calls: list[dict]) -> list[dict]:
    """Drop exact repeats of an identical successful call (same name + args)."""
    seen, out = set(), []
    for c in calls:
        key = (c["name"], json.dumps(c["args"], sort_keys=True, default=str))
        if key not in seen:
            seen.add(key)
            out.append(c)
    return out


def match_calls(expected: list[dict], got: list[dict], ordered: bool) -> tuple[bool, bool]:
    """(selection_ok, args_ok) for the successful calls of a run."""
    got = _dedupe(got)
    exp_names = [e["name"] for e in expected]
    got_names = [g["name"] for g in got]
    if ordered:
        if got_names != exp_names:
            return False, False
        return True, all(match_args(e["name"], e["args"], g["args"]) for e, g in zip(expected, got))
    if Counter(got_names) != Counter(exp_names):
        return False, False
    for perm in itertools.permutations(got):
        if all(e["name"] == g["name"] and match_args(e["name"], e["args"], g["args"])
               for e, g in zip(expected, perm)):
            return True, True
    return True, False


# --------------------------------------------------------------------------
# Final-answer check
# --------------------------------------------------------------------------

_NUM_RE = re.compile(r"-?\d[\d,]*(?:\.\d+)?")


def _numbers_in(text: str) -> list[float]:
    out = []
    for m in _NUM_RE.finditer(text or ""):
        try:
            out.append(float(m.group(0).replace(",", "")))
        except ValueError:
            pass
    return out


def _answer_item_ok(spec, text: str) -> bool:
    if isinstance(spec, (int, float)) and not isinstance(spec, bool):
        return any(math.isclose(n, float(spec), rel_tol=ANSWER_REL_TOL, abs_tol=0.01)
                   for n in _numbers_in(text))
    return _norm_str(spec) in _norm_str(text or "")


def answer_correct(spec, text: str):
    if spec is None:
        return None
    if isinstance(spec, dict) and "$all" in spec:
        return all(_answer_item_ok(s, text) for s in spec["$all"])
    if isinstance(spec, list):
        return any(_answer_item_ok(s, text) for s in spec)
    return _answer_item_ok(spec, text)


# --------------------------------------------------------------------------
# Scoring one run + failure taxonomy
# --------------------------------------------------------------------------

# Our fine-grained labels -> the four README classes (others are extra classes
# that the data showed up and the README did not list).
README_CLASS = {
    "schema_hallucination": "1. Schema hallucination",
    "type_drift": "2. Type drift & parsing",
    "parse_failure": "2. Type drift & parsing",
    "invalid_type_or_enum": "2. Type drift & parsing",
    "argument_omission": "3. Argument omission",
    "cyclic_stagnation": "4. Cyclic stagnation",
}

# Order used to pick ONE primary label per failed run (most specific first)
PRIMARY_ORDER = [
    "cyclic_stagnation", "schema_hallucination", "parse_failure", "argument_omission",
    "invalid_type_or_enum", "unneeded_call", "tool_skipped", "wrong_tool",
    "incomplete_chain", "wrong_value", "invalid_value", "stopped_on_error", "answer_wrong",
]


def _repeat_loop(attempts: list[dict], k: int = 3) -> bool:
    keys = [(a.get("name"), json.dumps(a.get("raw_args"), sort_keys=True, default=str))
            for a in attempts]
    return any(n >= k for n in Counter(keys).values())


def score_run(task: dict, run: dict) -> dict:
    """Turn one trace into per-run metrics + taxonomy labels."""
    attempts = run.get("attempts", [])
    expected = task["expected_calls"]
    is_no_tool = not expected
    loop_mode = run["mode"] != "single_shot"

    ok_calls = [{"name": a["name"], "args": a["args"]} for a in attempts if a["status"] == "ok"]
    n_attempts = len(attempts)
    n_malformed = sum(a["status"] in MALFORMED for a in attempts)
    n_tool_err = sum(a["status"] == "tool_error" for a in attempts)
    drift_attempts = [a for a in attempts if a.get("type_drift")]

    if is_no_tool:
        selection_ok = n_attempts == 0
        args_ok = None
    else:
        selection_ok, args_ok = match_calls(expected, ok_calls, task["ordered"])
        if not selection_ok:
            args_ok = None  # argument correctness is conditional on selection
    exact = bool(selection_ok and (args_ok if not is_no_tool else True))

    finished = run.get("stop_reason") in (None, "final_answer")
    ans = answer_correct(task["expected_answer"], run.get("final_text", "")) \
        if (loop_mode or is_no_tool) else None
    if loop_mode:
        success = bool(exact and finished and ans is not False)
    else:
        success = None  # single-shot never sees tool results -> no end-to-end answer

    # ---------------- taxonomy (multi-label)
    labels = set()
    st = Counter(a["status"] for a in attempts)
    if st["unknown_tool"] or st["unknown_args"]:
        labels.add("schema_hallucination")
    if st["parse_error"]:
        labels.add("parse_failure")
    if st["missing_args"]:
        labels.add("argument_omission")
    if drift_attempts:
        labels.add("type_drift")
    if st["validation_error"]:
        labels.add("invalid_type_or_enum")
    if run.get("stop_reason") == "recursion_limit" or _repeat_loop(attempts):
        labels.add("cyclic_stagnation")
    if run.get("stop_reason") == "tool_error_no_feedback":
        labels.add("stopped_on_error")
    if is_no_tool and n_attempts:
        labels.add("unneeded_call")
    if not is_no_tool:
        exp_names = [e["name"] for e in expected]
        got_names = [c["name"] for c in _dedupe(ok_calls)]
        if n_attempts == 0:
            labels.add("tool_skipped")
        elif not selection_ok:
            if got_names and got_names == exp_names[:len(got_names)]:
                labels.add("incomplete_chain")   # right start, stopped too early
            elif got_names:
                labels.add("wrong_tool")         # valid calls, but wrong tool(s)/order/count
        elif args_ok is False:
            labels.add("wrong_value")            # right tools, wrong argument values
    if st["tool_error"]:
        labels.add("invalid_value")              # schema-valid value the tool rejected
    if loop_mode and exact and finished and ans is False:
        labels.add("answer_wrong")

    failed = (not success) if loop_mode else (not exact)
    primary = None
    if failed:  # recovered runs keep their labels but get no primary failure
        primary = next((lab for lab in PRIMARY_ORDER if lab in labels), "other")

    return {
        "model": run["model"], "mode": run["mode"], "task_id": task["id"],
        "category": task["category"],
        "selection_ok": selection_ok, "args_ok": args_ok, "exact_match": exact,
        "answer_ok": ans, "success": success, "finished": finished,
        "stop_reason": run.get("stop_reason"),
        "n_attempts": n_attempts, "n_malformed": n_malformed, "n_tool_error": n_tool_err,
        "n_type_drift": len(drift_attempts),
        "any_malformed": n_malformed > 0,
        "recovered": bool(success and n_malformed + n_tool_err > 0),
        "answer_ok_but_call_wrong": bool(ans is True and not exact and not is_no_tool),
        "llm_calls": len(run.get("turns", [])),
        "latency_s": round(sum(t.get("latency_s", 0) for t in run.get("turns", [])), 3),
        "output_tokens": sum(t.get("eval_count") or 0 for t in run.get("turns", [])),
        "labels": sorted(labels), "primary_failure": primary,
    }

In [ ]:
%%writefile reliability_eval/agents.py
"""The two ways of running a task (plus one ablation).

single_shot             one model call; every tool call must be in that one
                        reply; the model never sees tool results.
react                   LangGraph ReAct loop: agent -> (tools -> agent)* -> END.
                        The tool node validates each call against the schema
                        and returns errors to the model as ToolMessages, so it
                        can correct itself.
react_no_error_feedback same graph, but the first invalid call ends the run.
                        This mimics the upstream tutorial's tool node, which
                        has no error handling (an unknown tool name raises
                        KeyError and the run crashes).

Every run returns a trace dict: attempts, per-turn latency/tokens, messages,
final text, stop reason.
"""

from __future__ import annotations

import json
import operator
import time
from typing import Annotated, Any

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from .scoring import execute_call, text_json_calls, validate_call
from .tools import TOOLS

MODES = ("single_shot", "react", "react_no_error_feedback")

BASE_PROMPT = (
    "You are a helpful assistant that can call tools.\n"
    "Use a tool whenever the request needs arithmetic, weather, currency conversion, "
    "employee directory data, or a reminder. Do not do arithmetic in your head and do "
    "not guess data. For general-knowledge questions or chit-chat, answer directly "
    "without calling any tool."
)
SINGLE_SHOT_SUFFIX = (
    "\nYou get exactly ONE response and will not see any tool results. Put every tool "
    "call the request needs into this single response."
)
REACT_SUFFIX = (
    "\nYou can work over several turns: call a tool, read its result, then call more "
    "tools or give the final answer. For multi-step requests, do one step at a time and "
    "use each result. When you are done, reply with a short final answer that states "
    "the result."
)


def make_llm(model: str, base_url: str, num_ctx: int = 4096, seed: int = 42):
    from langchain_ollama import ChatOllama
    return ChatOllama(
        model=model, base_url=base_url, temperature=0.0, num_ctx=num_ctx,
        seed=seed, num_predict=512, keep_alive="30m",
    )


# --------------------------------------------------------------------------
# helpers
# --------------------------------------------------------------------------

def _invoke_with_retry(llm, messages, retries: int = 3):
    last = None
    for i in range(retries):
        try:
            t0 = time.perf_counter()
            msg = llm.invoke(messages)
            return msg, time.perf_counter() - t0
        except Exception as e:  # connection hiccup, server restart ...
            last = e
            time.sleep(2 * (i + 1))
    raise last


def _turn_info(msg: AIMessage, latency: float) -> dict:
    md = getattr(msg, "response_metadata", {}) or {}
    return {
        "latency_s": round(latency, 3),
        "eval_count": md.get("eval_count"),             # output tokens
        "prompt_eval_count": md.get("prompt_eval_count"),  # input tokens
        "n_tool_calls": len(msg.tool_calls or []),
    }


def _parse_failures(msg: AIMessage, turn: int) -> list[dict]:
    """Attempts that never became structured calls."""
    out = []
    for bad in getattr(msg, "invalid_tool_calls", None) or []:
        out.append({"turn": turn, "source": "invalid_tool_call", "name": bad.get("name"),
                    "raw_args": bad.get("args"), "status": "parse_error",
                    "error": bad.get("error") or "could not parse tool call", "args": None,
                    "type_drift": []})
    if not msg.tool_calls:  # JSON written into the text instead of a real call
        for name in text_json_calls(msg.content if isinstance(msg.content, str) else str(msg.content)):
            out.append({"turn": turn, "source": "text_json", "name": name, "raw_args": None,
                        "status": "parse_error",
                        "error": "tool call written as text instead of a structured call",
                        "args": None, "type_drift": []})
    return out


def _check_and_run(tc: dict, turn: int) -> dict:
    v = validate_call(tc["name"], tc.get("args"))
    att = {"turn": turn, "source": "structured", "name": tc["name"], "raw_args": tc.get("args"),
           "status": v["status"], "error": v["error"], "args": v["args"],
           "type_drift": v["type_drift"], "result": None}
    if v["status"] == "ok":
        status, result, err = execute_call(tc["name"], v["args"])
        att.update(status=status, result=result, error=err)
    return att


def _ser(m) -> dict:
    d = {"type": m.type, "content": m.content if isinstance(m.content, str) else str(m.content)}
    if isinstance(m, AIMessage) and m.tool_calls:
        d["tool_calls"] = [{"name": t["name"], "args": t["args"]} for t in m.tool_calls]
    if isinstance(m, ToolMessage):
        d["name"], d["status"] = m.name, getattr(m, "status", None)
    return d


# --------------------------------------------------------------------------
# single-shot
# --------------------------------------------------------------------------

def run_single_shot(llm_tools, prompt: str) -> dict:
    messages = [SystemMessage(BASE_PROMPT + SINGLE_SHOT_SUFFIX), HumanMessage(prompt)]
    msg, lat = _invoke_with_retry(llm_tools, messages)
    attempts = [_check_and_run(tc, 0) for tc in msg.tool_calls or []] + _parse_failures(msg, 0)
    return {
        "attempts": attempts, "turns": [_turn_info(msg, lat)],
        "final_text": msg.content if isinstance(msg.content, str) else str(msg.content),
        "stop_reason": "final_answer", "messages": [_ser(m) for m in messages[1:] + [msg]],
    }


# --------------------------------------------------------------------------
# ReAct on LangGraph
# --------------------------------------------------------------------------

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]          # chat history (reducer appends)
    attempts: Annotated[list, operator.add]          # every tool-call attempt
    turns: Annotated[list, operator.add]             # per-LLM-call timing/tokens
    stop: str | None                                 # why the run ended early


def build_react_graph(llm_tools, error_feedback: bool = True):
    system = SystemMessage(BASE_PROMPT + REACT_SUFFIX)

    def agent(state: AgentState) -> dict:
        msg, lat = _invoke_with_retry(llm_tools, [system] + state["messages"])
        turn = len(state["turns"])
        return {"messages": [msg], "turns": [_turn_info(msg, lat)],
                "attempts": _parse_failures(msg, turn)}

    def tools(state: AgentState) -> dict:
        last = state["messages"][-1]
        turn = len(state["turns"]) - 1
        attempts, outs, stop = [], [], None
        for tc in last.tool_calls:
            att = _check_and_run(tc, turn)
            attempts.append(att)
            if att["status"] == "ok":
                outs.append(ToolMessage(content=json.dumps(att["result"], default=str),
                                        name=tc["name"], tool_call_id=tc["id"]))
            else:
                outs.append(ToolMessage(content=f"Error: {att['error']} Please fix the call and try again.",
                                        name=tc["name"], tool_call_id=tc["id"], status="error"))
                if not error_feedback:
                    stop = "tool_error_no_feedback"
        return {"messages": outs, "attempts": attempts, "stop": stop}

    def after_agent(state: AgentState) -> str:
        last = state["messages"][-1]
        return "tools" if getattr(last, "tool_calls", None) else END

    def after_tools(state: AgentState) -> str:
        return END if state.get("stop") else "agent"

    g = StateGraph(AgentState)
    g.add_node("agent", agent)
    g.add_node("tools", tools)
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", after_agent, {"tools": "tools", END: END})
    g.add_conditional_edges("tools", after_tools, {"agent": "agent", END: END})
    return g.compile()


def run_react(graph, prompt: str, recursion_limit: int = 25) -> dict:
    state: dict[str, Any] = {"messages": [HumanMessage(prompt)], "attempts": [], "turns": [], "stop": None}
    last = state
    stop = None
    try:
        for last in graph.stream(state, config={"recursion_limit": recursion_limit},
                                 stream_mode="values"):
            pass
    except GraphRecursionError:
        stop = "recursion_limit"
    stop = stop or last.get("stop") or "final_answer"
    msgs = last["messages"]
    final = ""
    if stop == "final_answer" and msgs and isinstance(msgs[-1], AIMessage):
        c = msgs[-1].content
        final = c if isinstance(c, str) else str(c)
    return {"attempts": last["attempts"], "turns": last["turns"], "final_text": final,
            "stop_reason": stop, "messages": [_ser(m) for m in msgs]}


class Runner:
    """Holds one model's tool-bound LLM and compiled graphs."""

    def __init__(self, llm, recursion_limit: int = 25):
        self.llm_tools = llm.bind_tools(TOOLS)
        self.recursion_limit = recursion_limit
        self.graphs = {
            "react": build_react_graph(self.llm_tools, error_feedback=True),
            "react_no_error_feedback": build_react_graph(self.llm_tools, error_feedback=False),
        }

    def run(self, mode: str, prompt: str) -> dict:
        t0 = time.perf_counter()
        if mode == "single_shot":
            out = run_single_shot(self.llm_tools, prompt)
        else:
            out = run_react(self.graphs[mode], prompt, self.recursion_limit)
        out["wall_s"] = round(time.perf_counter() - t0, 3)
        return out

In [ ]:
%%writefile reliability_eval/stats.py
"""Small, dependency-free statistics used in the report.

wilson(k, n)       95 % Wilson score interval for a proportion k/n.
                   Better than "p +/- 1.96*sqrt(p(1-p)/n)" for small n or
                   p near 0 / 1 (it never goes below 0 or above 1).
mcnemar(b, c)      exact two-sided McNemar test for PAIRED yes/no outcomes
                   (same tasks, two systems). b = tasks only system A got
                   right, c = tasks only system B got right. Tasks both got
                   right / both got wrong carry no information about the
                   difference and are ignored.
"""

from __future__ import annotations

import math


def wilson(k: int, n: int, z: float = 1.959964) -> tuple[float, float]:
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))


def mcnemar(b: int, c: int) -> float:
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * tail)


def fmt_pct(k: int, n: int) -> str:
    if n == 0:
        return "n/a"
    lo, hi = wilson(k, n)
    return f"{100 * k / n:.1f}% ({k}/{n}) [{100 * lo:.0f}–{100 * hi:.0f}]"

In [ ]:
%%writefile reliability_eval/run_eval.py
"""Run the benchmark against models served by Ollama.

    python -m reliability_eval.run_eval --split test \
        --models llama3.2:3b qwen2.5:3b qwen2.5:7b llama3.2:1b \
        --modes single_shot react react_no_error_feedback --out results

Resumable: runs already in <out>/traces.jsonl are skipped.
"""

from __future__ import annotations

import argparse
import json
import platform
import subprocess
import sys
import time
import urllib.request
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

from .agents import MODES, Runner, make_llm
from .tasks import load_tasks


def _get(url: str, data: dict | None = None):
    req = urllib.request.Request(url, data=json.dumps(data).encode() if data else None,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())


def environment(base_url: str, models: list[str]) -> dict:
    env = {"timestamp_utc": datetime.now(timezone.utc).isoformat(),
           "python": sys.version.split()[0], "platform": platform.platform(), "packages": {}}
    for p in ("langgraph", "langchain-core", "langchain-ollama", "ollama", "pydantic"):
        try:
            env["packages"][p] = version(p)
        except PackageNotFoundError:
            env["packages"][p] = None
    try:
        env["ollama_version"] = _get(f"{base_url}/api/version").get("version")
    except Exception as e:
        env["ollama_version"] = f"unavailable: {e}"
    env["models"] = {}
    tags = {}
    try:
        tags = {m["name"]: m for m in _get(f"{base_url}/api/tags").get("models", [])}
    except Exception:
        pass
    for m in models:
        info = {"digest": tags.get(m, {}).get("digest"), "size_bytes": tags.get(m, {}).get("size")}
        try:
            d = _get(f"{base_url}/api/show", {"model": m}).get("details", {})
            info.update(parameter_size=d.get("parameter_size"), quantization=d.get("quantization_level"),
                        family=d.get("family"), format=d.get("format"))
        except Exception:
            pass
        env["models"][m] = info
    try:
        env["gpu"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10).stdout.strip()
    except Exception:
        env["gpu"] = None
    return env


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="+", default=["llama3.2:3b", "qwen2.5:7b"])
    ap.add_argument("--modes", nargs="+", default=list(MODES), choices=MODES)
    ap.add_argument("--split", default="test", choices=["dev", "test", "all"])
    ap.add_argument("--tasks", nargs="*", help="only these task ids")
    ap.add_argument("--out", default="results")
    ap.add_argument("--base-url", default="http://127.0.0.1:11434")
    ap.add_argument("--recursion-limit", type=int, default=25)  # same as repo config.py
    ap.add_argument("--num-ctx", type=int, default=4096)        # same as repo config.py
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args(argv)

    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    tasks = load_tasks(a.split)
    if a.tasks:
        tasks = [t for t in tasks if t["id"] in set(a.tasks)]
    trace_path = out / "traces.jsonl"
    done = set()
    if trace_path.exists():
        for line in trace_path.read_text().splitlines():
            r = json.loads(line)
            done.add((r["model"], r["mode"], r["task_id"]))

    env = environment(a.base_url, a.models)
    env.update(split=a.split, n_tasks=len(tasks), modes=a.modes, recursion_limit=a.recursion_limit,
               num_ctx=a.num_ctx, seed=a.seed, temperature=0.0)
    (out / f"env_{int(time.time())}.json").write_text(json.dumps(env, indent=2))
    print(json.dumps(env, indent=2))

    t_all = time.perf_counter()
    with open(trace_path, "a") as f:
        for model in a.models:
            runner = Runner(make_llm(model, a.base_url, a.num_ctx, a.seed), a.recursion_limit)
            # warm-up so the first timed task does not include model loading
            t0 = time.perf_counter()
            runner.llm_tools.invoke("Say OK.")
            print(f"[{model}] loaded in {time.perf_counter() - t0:.1f}s", flush=True)
            for mode in a.modes:
                t_mode = time.perf_counter()
                for i, task in enumerate(tasks, 1):
                    if (model, mode, task["id"]) in done:
                        continue
                    try:
                        tr = runner.run(mode, task["prompt"])
                    except Exception as e:  # never lose the whole run to one task
                        tr = {"attempts": [], "turns": [], "final_text": "",
                              "stop_reason": f"exception: {type(e).__name__}: {e}", "messages": []}
                    tr.update(model=model, mode=mode, task_id=task["id"], split=task["split"])
                    f.write(json.dumps(tr, default=str) + "\n")
                    f.flush()
                    calls = [x["name"] for x in tr["attempts"]]
                    print(f"[{model}|{mode}] {i}/{len(tasks)} {task['id']:7s} "
                          f"{tr.get('wall_s', 0):6.2f}s stop={tr['stop_reason']} calls={calls}",
                          flush=True)
                print(f"[{model}|{mode}] done in {time.perf_counter() - t_mode:.0f}s", flush=True)
    print(f"ALL DONE in {time.perf_counter() - t_all:.0f}s")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile reliability_eval/report.py
"""Score traces and write the report.

    python -m reliability_eval.report --out results [--split test]

Writes  scores.csv, summary.json, summary.md, examples.md  into --out.
"""

from __future__ import annotations

import argparse
import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

from .scoring import README_CLASS, score_run
from .stats import fmt_pct, mcnemar
from .tasks import load_tasks

MODE_ORDER = ["single_shot", "react", "react_no_error_feedback"]
CATS = ["single_simple", "single_args", "multi_step", "no_tool"]


def load(out: Path, split: str):
    tasks = {t["id"]: t for t in load_tasks("all")}
    runs = {}
    for line in (out / "traces.jsonl").read_text().splitlines():
        r = json.loads(line)
        if split != "all" and r.get("split") != split:
            continue
        runs[(r["model"], r["mode"], r["task_id"])] = r  # last write wins
    scores = [score_run(tasks[k[2]], r) for k, r in runs.items()]
    return tasks, runs, scores


def _k(rows, key, val=True):
    return sum(1 for r in rows if r[key] == val)


def aggregate(rows):
    n = len(rows)
    tool_rows = [r for r in rows if r["category"] != "no_tool"]
    sel_ok = [r for r in tool_rows if r["selection_ok"]]
    att = sum(r["n_attempts"] for r in rows)
    mal = sum(r["n_malformed"] for r in rows)
    loop = rows and rows[0]["mode"] != "single_shot"
    d = {
        "n_tasks": n,
        "selection": (_k(rows, "selection_ok"), n),
        "args_given_selection": (_k(sel_ok, "args_ok"), len(sel_ok)),
        "exact": (_k(rows, "exact_match"), n),
        "exact_tool_tasks": (_k(tool_rows, "exact_match"), len(tool_rows)),
        "malformed_attempts": (mal, att),
        "tasks_with_malformed": (_k(rows, "any_malformed"), n),
        "tool_errors": sum(r["n_tool_error"] for r in rows),
        "type_drift_attempts": sum(r["n_type_drift"] for r in rows),
        "success": (_k(rows, "success"), n) if loop else None,
        "recovered": _k(rows, "recovered") if loop else None,
        "answer_ok_but_call_wrong": _k(rows, "answer_ok_but_call_wrong"),
        "mean_latency_s": round(sum(r["latency_s"] for r in rows) / max(n, 1), 2),
        "total_latency_s": round(sum(r["latency_s"] for r in rows), 1),
        "mean_llm_calls": round(sum(r["llm_calls"] for r in rows) / max(n, 1), 2),
        "stop_reasons": dict(Counter(r["stop_reason"] for r in rows)),
    }
    return d


def paired(rows_a, rows_b, key):
    a = {r["task_id"]: bool(r[key]) for r in rows_a}
    b = {r["task_id"]: bool(r[key]) for r in rows_b}
    ids = sorted(set(a) & set(b))
    both = sum(a[i] and b[i] for i in ids)
    only_a = sum(a[i] and not b[i] for i in ids)
    only_b = sum(b[i] and not a[i] for i in ids)
    neither = len(ids) - both - only_a - only_b
    return {"n": len(ids), "a_right": both + only_a, "b_right": both + only_b, "both": both,
            "only_a": only_a, "only_b": only_b, "neither": neither,
            "p_mcnemar": round(mcnemar(only_a, only_b), 4)}


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="results")
    ap.add_argument("--split", default="test")
    a = ap.parse_args(argv)
    out = Path(a.out)
    tasks, runs, scores = load(out, a.split)
    if not scores:
        raise SystemExit("no runs found for this split")

    with open(out / "scores.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(scores[0]))
        w.writeheader()
        for s in scores:
            w.writerow({**s, "labels": "|".join(s["labels"])})

    groups = defaultdict(list)
    for s in scores:
        groups[(s["model"], s["mode"])].append(s)
    models = list(dict.fromkeys(s["model"] for s in scores))
    keys = [(m, md) for m in models for md in MODE_ORDER if (m, md) in groups]
    agg = {f"{m}|{md}": aggregate(groups[(m, md)]) for m, md in keys}

    L = [f"# Tool-calling reliability results ({a.split} split)\n"]
    n_tasks = len({s['task_id'] for s in scores})
    cat_n = Counter(tasks[t]["category"] for t in {s['task_id'] for s in scores})
    L.append(f"Tasks: **{n_tasks}** ({', '.join(f'{c} {cat_n[c]}' for c in CATS)}). "
             "Temperature 0. Cells show `% (k/n) [95% Wilson CI]`.\n")

    L.append("## 1. Main table\n")
    L.append("| Model | Mode | Tool selection acc. | Arg. correctness (given right tools) | "
             "Exact call match | Malformed call rate (per attempt) | Tasks with ≥1 malformed | "
             "End-to-end success | Mean s/task | Mean LLM calls |")
    L.append("|---|---|---|---|---|---|---|---|---|---|")
    for m, md in keys:
        g = agg[f"{m}|{md}"]
        succ = fmt_pct(*g["success"]) if g["success"] else "— (no final answer in single-shot)"
        L.append(f"| {m} | {md} | {fmt_pct(*g['selection'])} | {fmt_pct(*g['args_given_selection'])} | "
                 f"{fmt_pct(*g['exact'])} | {fmt_pct(*g['malformed_attempts'])} | "
                 f"{fmt_pct(*g['tasks_with_malformed'])} | {succ} | {g['mean_latency_s']} | "
                 f"{g['mean_llm_calls']} |")
    L.append("\nDefinitions: selection = the set/sequence of successfully executed tool names equals "
             "the ground truth (for no_tool tasks: no call at all). Argument correctness is measured "
             "only on tasks where selection was right. Exact = both. Malformed = unknown tool, "
             "unknown/missing argument, uncoercible type/enum, or unparseable / text-only call. "
             "Success (loop modes) = exact match AND the run finished AND the final answer "
             "contains the expected value.\n")

    L.append("## 2. Exact call match by category\n")
    L.append("| Model | Mode | " + " | ".join(CATS) + " |")
    L.append("|---|---|" + "---|" * len(CATS))
    for m, md in keys:
        cells = []
        for c in CATS:
            rows = [r for r in groups[(m, md)] if r["category"] == c]
            cells.append(fmt_pct(_k(rows, "exact_match"), len(rows)))
        L.append(f"| {m} | {md} | " + " | ".join(cells) + " |")

    L.append("\n## 3. Paired comparisons (same tasks)\n")
    L.append("`only A` / `only B` = tasks exactly one system got right; only these drive the "
             "exact McNemar p-value. A gap of g points on n tasks = g·n/100 tasks.\n")
    L.append("| Comparison | Metric | Subset | n | A right | B right | only A | only B | gap (pts) | p |")
    L.append("|---|---|---|---|---|---|---|---|---|---|")
    comps = []
    for m in models:
        for x, y, metric in [("react", "single_shot", "exact_match"),
                             ("react", "react_no_error_feedback", "success")]:
            if (m, x) in groups and (m, y) in groups:
                comps.append((f"{m}: {x} (A) vs {y} (B)", groups[(m, x)], groups[(m, y)], metric))
    for x, y in [("qwen2.5:7b", "qwen2.5:3b"), ("llama3.2:3b", "llama3.2:1b"),
                 ("qwen2.5:3b", "llama3.2:3b"), ("qwen2.5:7b", "llama3.2:3b")]:
        for md in MODE_ORDER:
            if (x, md) in groups and (y, md) in groups:
                comps.append((f"{md}: {x} (A) vs {y} (B)", groups[(x, md)], groups[(y, md)],
                              "success" if md != "single_shot" else "exact_match"))
    pair_json = []
    for name, ra, rb, metric in comps:
        for subset, f in [("all", lambda r: True),
                          ("excluding multi_step", lambda r: r["category"] != "multi_step")]:
            p = paired([r for r in ra if f(r)], [r for r in rb if f(r)], metric)
            gap = 100 * (p["a_right"] - p["b_right"]) / max(p["n"], 1)
            L.append(f"| {name} | {metric} | {subset} | {p['n']} | {p['a_right']} | {p['b_right']} | "
                     f"{p['only_a']} | {p['only_b']} | {gap:+.1f} | {p['p_mcnemar']} |")
            pair_json.append({"comparison": name, "metric": metric, "subset": subset, **p,
                              "gap_pts": round(gap, 1)})

    L.append("\n## 4. Failure taxonomy (primary label per failed run)\n")
    labels = sorted({s["primary_failure"] for s in scores if s["primary_failure"]})
    L.append("| Model | Mode | failed runs | " + " | ".join(labels) + " |")
    L.append("|---|---|---|" + "---|" * len(labels))
    tax = {}
    for m, md in keys:
        c = Counter(r["primary_failure"] for r in groups[(m, md)] if r["primary_failure"])
        tax[f"{m}|{md}"] = dict(c)
        L.append(f"| {m} | {md} | {sum(c.values())} | " + " | ".join(str(c.get(l, 0)) for l in labels) + " |")
    allc = Counter(s["primary_failure"] for s in scores if s["primary_failure"])
    L.append("\nAll runs pooled: " + ", ".join(f"{k} {v}" for k, v in allc.most_common()))
    readme = Counter()
    for s in scores:
        for lab in s["labels"]:
            readme[README_CLASS.get(lab, f"(extra) {lab}")] += 1
    L.append("\nMulti-label counts grouped into the README's four classes (+ extra classes found):\n")
    for k, v in sorted(readme.items()):
        L.append(f"- {k}: {v}")

    L.append("\n## 5. Recovery and 'answer looked right' (loop modes)\n")
    for m, md in keys:
        g = agg[f"{m}|{md}"]
        L.append(f"- {m} | {md}: recovered after an error = {g['recovered']}; final answer contained the "
                 f"expected value but the tool call was wrong = {g['answer_ok_but_call_wrong']}; "
                 f"type-drift attempts (coerced) = {g['type_drift_attempts']}; tool errors = {g['tool_errors']}; "
                 f"stop reasons = {g['stop_reasons']}")

    L.append("\n## 6. Run time\n")
    for m, md in keys:
        g = agg[f"{m}|{md}"]
        L.append(f"- {m} | {md}: {g['total_latency_s']} s model time over {g['n_tasks']} tasks "
                 f"(mean {g['mean_latency_s']} s/task)")
    envs = sorted(out.glob("env_*.json"))
    if envs:
        e = json.loads(envs[-1].read_text())
        L.append(f"\nEnvironment: GPU {e.get('gpu')}; Ollama {e.get('ollama_version')}; "
                 f"packages {e.get('packages')}; models {json.dumps(e.get('models'))}")

    (out / "summary.md").write_text("\n".join(L) + "\n")
    (out / "summary.json").write_text(json.dumps({"aggregate": agg, "paired": pair_json,
                                                  "taxonomy_primary": tax}, indent=2, default=str))

    # ---- examples for interview stories
    E = ["# Example failures (up to 2 per label)\n"]
    by_label = defaultdict(list)
    for s in scores:
        if s["primary_failure"]:
            by_label[s["primary_failure"]].append(s)
    for lab, rows in sorted(by_label.items()):
        E.append(f"## {lab} ({len(rows)} runs)\n")
        for s in rows[:2]:
            t = tasks[s["task_id"]]
            r = runs[(s["model"], s["mode"], s["task_id"])]
            E.append(f"**{s['model']} | {s['mode']} | {t['id']}**: {t['prompt']}")
            E.append(f"- expected: `{json.dumps(t['expected_calls'])}`")
            for at in r["attempts"]:
                E.append(f"- attempt: `{at.get('name')}({json.dumps(at.get('raw_args'), default=str)})` "
                         f"→ {at['status']}" + (f" ({at['error'][:120]})" if at.get("error") else ""))
            E.append(f"- final text: {r.get('final_text', '')[:200]!r}; stop: {r['stop_reason']}\n")
    (out / "examples.md").write_text("\n".join(E) + "\n")
    print((out / "summary.md").read_text())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile reliability_eval/test_harness.py
"""Offline tests for the harness (no Ollama needed): a scripted fake model
plays back fixed replies so every scoring path is exercised.

    python -m pytest reliability_eval/test_harness.py -q
"""

from __future__ import annotations

import itertools

from langchain_core.messages import AIMessage

from reliability_eval.agents import Runner
from reliability_eval.scoring import match_calls, score_run, validate_call
from reliability_eval.tasks import load_tasks

TASKS = {t["id"]: t for t in load_tasks("all")}
_ids = itertools.count()


def call(tool_name, **args):
    return {"name": tool_name, "args": args, "id": f"c{next(_ids)}", "type": "tool_call"}


class FakeLLM:
    """Replies come from a script: list of AIMessages, consumed in order."""

    def __init__(self, script):
        self.script = list(script)

    def bind_tools(self, tools):
        return self

    def invoke(self, messages):
        if not self.script:
            return AIMessage(content="done")
        nxt = self.script.pop(0)
        return nxt if not callable(nxt) else nxt(messages)


def run(task_id, mode, script, limit=25):
    r = Runner(FakeLLM(script), recursion_limit=limit).run(mode, TASKS[task_id]["prompt"])
    r.update(model="fake", mode=mode, task_id=task_id)
    return score_run(TASKS[task_id], r), r


def test_validate_paths():
    assert validate_call("add", {"a": 1, "b": 2})["status"] == "ok"
    assert validate_call("power", {"a": 1})["status"] == "unknown_tool"
    assert validate_call("add", {"x": 1, "b": 2})["status"] == "unknown_args"
    assert validate_call("add", {"a": 1})["status"] == "missing_args"
    assert validate_call("get_weather", {"city": "Pune", "unit": "F"})["status"] == "validation_error"
    v = validate_call("add", {"a": "25", "b": 4})
    assert v["status"] == "ok" and v["type_drift"] == ["a"] and v["args"]["a"] == 25.0


def test_single_shot_correct_and_commutative():
    s, _ = run("s01", "single_shot", [AIMessage("", tool_calls=[call("multiply", a=4, b=25)])])
    assert s["selection_ok"] and s["args_ok"] and s["exact_match"] and s["success"] is None


def test_subtract_order_matters():
    s, _ = run("s04", "single_shot", [AIMessage("", tool_calls=[call("subtract", a=1999, b=5000)])])
    assert s["selection_ok"] and s["args_ok"] is False and s["primary_failure"] == "wrong_value"


def test_tool_skipped_but_answer_right():
    s, _ = run("s01", "react", [AIMessage("25 times 4 is 100.")])
    assert not s["exact_match"] and s["answer_ok"] and s["answer_ok_but_call_wrong"]
    assert s["primary_failure"] == "tool_skipped"


def test_react_multistep_with_recovery():
    script = [
        AIMessage("", tool_calls=[call("sum", a=15, b=27)]),        # hallucinated tool
        AIMessage("", tool_calls=[call("add", a=15, b=27)]),
        AIMessage("", tool_calls=[call("multiply", a=42, b=3)]),
        AIMessage("The answer is 126."),
    ]
    s, r = run("m01", "react", script)
    assert s["success"] and s["recovered"] and s["n_malformed"] == 1
    assert "schema_hallucination" in s["labels"] and s["primary_failure"] is None
    # same script without error feedback stops at the first bad call
    s2, r2 = run("m01", "react_no_error_feedback", script)
    assert not s2["success"] and r2["stop_reason"] == "tool_error_no_feedback"


def test_recursion_limit_is_stagnation():
    loop = [AIMessage("", tool_calls=[call("get_weather", city="Pune", unit="F")]) for _ in range(30)]
    s, r = run("s07", "react", loop, limit=8)
    assert r["stop_reason"] == "recursion_limit" and s["primary_failure"] == "cyclic_stagnation"


def test_text_json_is_parse_failure():
    s, _ = run("s01", "single_shot",
               [AIMessage('{"name": "multiply", "parameters": {"a": 25, "b": 4}}')])
    assert s["n_malformed"] == 1 and s["primary_failure"] == "parse_failure"


def test_no_tool_task():
    s, _ = run("n01", "react", [AIMessage("The capital of France is Paris.")])
    assert s["exact_match"] and s["success"]
    s, _ = run("n01", "react", [AIMessage("", tool_calls=[call("get_weather", city="Paris")]),
                                AIMessage("Paris.")])
    assert not s["exact_match"] and s["primary_failure"] == "unneeded_call"


def test_unordered_and_incomplete_chain():
    exp = TASKS["m15"]["expected_calls"]
    got = [{"name": "get_weather", "args": {"city": "Delhi", "unit": "celsius"}},
           {"name": "get_weather", "args": {"city": "Bangalore", "unit": "celsius"}}]
    assert match_calls(exp, got, ordered=False) == (True, True)
    s, _ = run("m11", "react", [AIMessage("", tool_calls=[call("find_employee", name="Kabir Singh")]),
                                AIMessage("Kabir's ID is 1050.")])
    assert s["primary_failure"] == "incomplete_chain"


def test_tool_error_value():
    s, _ = run("s11", "react", [
        AIMessage("", tool_calls=[call("convert_currency", amount=250, from_currency="dollars",
                                       to_currency="rupees")]),
        AIMessage("", tool_calls=[call("convert_currency", amount=250, from_currency="USD",
                                       to_currency="INR")]),
        AIMessage("That is 20,750 rupees."),
    ])
    assert s["success"] and s["recovered"] and s["n_tool_error"] == 1 and s["n_malformed"] == 0

## 4. Offline tests of the harness (scripted fake model, no LLM)

In [ ]:
!python -m pytest reliability_eval/test_harness.py -q -p no:cacheprovider
!python -m reliability_eval.tasks

## 5. Dev-split smoke test (12 tasks, first model only). Not used in reported numbers.

In [ ]:
!python -m reliability_eval.run_eval --split dev --models llama3.2:3b --modes single_shot react --out results_dev
!python -m reliability_eval.report --out results_dev --split dev > /dev/null && head -20 results_dev/summary.md

## 6. Full held-out test run (72 tasks × 3 modes × all models)

In [ ]:
t0 = time.time()
!python -m reliability_eval.run_eval --split test --models {MODELS_STR} --out results
print(f"test run wall time: {(time.time()-t0)/60:.1f} min")

## 7. Report

In [ ]:
!python -m reliability_eval.report --out results --split test > /dev/null
from IPython.display import Markdown, display
display(Markdown((OUT / "summary.md").read_text()))

In [ ]:
!cp reliability_eval/tasks.jsonl results/ 2>/dev/null; cp ollama_server.log results/ 2>/dev/null
!cd /kaggle/working 2>/dev/null; zip -qr results.zip results results_dev && ls -lh results.zip